# MetaAlgorithmGA Testing on 1K-Node Clustered Graph

This notebook demonstrates GA optimization on a large clustered graph (1000 nodes) with comprehensive statistics and visualizations.

## Setup & Imports

In [7]:
import sys
import time
import matplotlib.pyplot as plt
import networkx as nx

from tests.fixtures.graphs import _create_clustered_graph

sys.path.insert(0, '..')

from meta.core import CanonicalVector, FitnessEvaluator, MetaAlgorithmGA, DistributedCascadingEvaluator
from src.graph.graph_manager import GraphManager
from src.meta.parameterizers.greedy import GreedyParameterizer
from src.meta.parameterizers.itai import ItaiParameterizer
from src.meta.parameterizers.luby import LubyParameterizer

## Helper Functions

In [8]:
def fixture_to_graph(fixture_dict) -> GraphManager:
    """Convert fixture dictionary to GraphManager."""
    graph = GraphManager.create_empty_graph()
    for v in fixture_dict['vertices']:
        graph.add_vertex(v)
    for u, v, w in fixture_dict['edges']:
        graph.add_edge(u, v, float(w))
    return graph

def format_time(seconds: float) -> str:
    """Format time in human-readable form."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    else:
        return f"{seconds/60:.1f}m"

def get_optimal_weight(fixture_dict) -> float:
    """Compute optimal matching weight using NetworkX."""
    try:
        G = nx.Graph()
        for v in fixture_dict['vertices']:
            G.add_node(v)
        for u, v, w in fixture_dict['edges']:
            G.add_edge(u, v, weight=float(w))
        matching = nx.max_weight_matching(G, weight='weight', maxcardinality=False)
        return sum(G[u][v].get('weight', 1.0) for u, v in matching)
    except Exception:
        return 0.0

def get_algorithm_baselines(graph: GraphManager, evaluator: FitnessEvaluator) -> dict:
    """Get baseline results from merged algorithms using FitnessEvaluator.

    Uses same evaluation path as GA to ensure fair comparison.
    """
    vector = CanonicalVector(
        luby_base_probability=0.5,
        luby_coeff_degree=0.0,
        luby_coeff_neighbors_unmatched=0.0,
        luby_coeff_clustering=0.0,
        luby_coeff_matched=0.0,
        luby_coeff_round=0.0,
        luby_coeff_weight=0.0,
        itai_timeout_rounds=5,
        max_iterations=10,
        convergence_threshold=0.05,
    )

    try:
        weight = evaluator.evaluate(graph, vector)
        return {'baseline': weight}
    except Exception:
        pass

    return {'baseline': 0.0}

## Define Graph Size and GA Parameters

In [9]:
POPULATION_SIZE = 20
GENERATIONS = 10
SEED = 45
NR_OF_NODES = 1000

## Test: GA Optimization on 1K-Node Clustered Graph

In [10]:
# Load graph
print("="*80)
print(f"LOADING {NR_OF_NODES}-NODE CLUSTERED GRAPH (Seed: {SEED})")
print("="*80)


fixture = _create_clustered_graph(nr_of_nudes=NR_OF_NODES, seed=SEED)
graph = fixture_to_graph(fixture)

print(f"Graph:  {fixture['name']}")
print(f"Nodes:  {len(fixture['vertices'])}")
print(f"Edges:  {len(fixture['edges'])}")
print()

# Get optimal weight
print("Computing optimal matching weight (NetworkX)...", end=" ", flush=True)
optimal = get_optimal_weight(fixture)
print(f"✓ {optimal:.0f}")

# Create evaluator (used for both baseline and GA)
print("Initializing fitness evaluator...", end=" ", flush=True)
evaluator = FitnessEvaluator()
print(f"✓ Done")

# Get baseline
print("Getting baseline from standard algorithms...", end=" ", flush=True)
baseline_result = get_algorithm_baselines(graph, evaluator)
baseline = baseline_result['baseline']
print(f"✓ {baseline:.0f}")
print()

LOADING 1000-NODE CLUSTERED GRAPH (Seed: 45)
Graph:  Clustered Graph with Communities (1000 nodes)
Nodes:  1000
Edges:  3945

Computing optimal matching weight (NetworkX)... ✓ 45337
Initializing fitness evaluator... ✓ Done
Getting baseline from standard algorithms... ✓ 39422



In [11]:
# Get cascading baseline
print("Getting cascading baseline (with distributed cascading)...", end=" ", flush=True)
cascading_evaluator = DistributedCascadingEvaluator()
baseline_vector = CanonicalVector(
    luby_base_probability=0.5,
    luby_coeff_degree=0.0,
    luby_coeff_neighbors_unmatched=0.0,
    luby_coeff_clustering=0.0,
    luby_coeff_matched=0.0,
    luby_coeff_round=0.0,
    luby_coeff_weight=0.0,
    itai_timeout_rounds=5,
    max_iterations=10,
    convergence_threshold=0.05,
)
baseline_cascading = cascading_evaluator.evaluate(graph, baseline_vector)
# Cascading details are stored in the evaluator object
cascades = cascading_evaluator.last_num_cascades
weights_per_cascade = cascading_evaluator.last_weights_per_cascade
print(f"✓ {baseline_cascading:.0f} ({cascades} cascades)")
print()

Getting cascading baseline (with distributed cascading)... ✓ 39516 (2 cascades)



## Run GA Optimization

In [12]:
print("="*80)
print("RUNNING GA OPTIMIZATION")
print("="*80)
print()

ga = MetaAlgorithmGA(
    fitness_evaluator=evaluator,
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    mutation_rate=0.15
)

print("GA Parameters:")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations:     {GENERATIONS}")
print(f"  Mutation rate:   0.15")
print()

print("Running GA...", flush=True)
start = time.time()
best_vector, fitness_history = ga.evolve(graph)
elapsed = time.time() - start

best = fitness_history[-1]
gap = ((optimal - best) / (optimal + 1e-10)) * 100
improvement = ((best - baseline) / (baseline + 1e-10)) * 100

print(f"✓ Done in {format_time(elapsed)}")
print()

RUNNING GA OPTIMIZATION

GA Parameters:
  Population size: 20
  Generations:     10
  Mutation rate:   0.15

Running GA...
✓ Done in 6.4m



## Results

In [13]:
print("="*80)
print("EVALUATOR COMPARISON: Standard vs Cascading")
print("="*80)
print()

print("Fitness by Evaluator:")
print(f"  Standard (no cascading):       {baseline:>12.0f}")
print(f"  Cascading (distributed):       {baseline_cascading:>12.0f} ({cascades} cascades)")
print(f"  Cascading improvement:         {((baseline_cascading - baseline) / baseline * 100):>11.1f}%")
print()

print("Cascading Details:")
print(f"  Weights per cascade round: {[f'{w:.0f}' for w in weights_per_cascade]}")
print()


EVALUATOR COMPARISON: Standard vs Cascading

Fitness by Evaluator:
  Standard (no cascading):              39422
  Cascading (distributed):              39516 (2 cascades)
  Cascading improvement:                 0.2%

Cascading Details:
  Weights per cascade round: ['39651', '39516']



In [14]:
print("="*80)
print("RESULTS SUMMARY")
print("="*80)
print()

print("Fitness Comparison:")
print(f"  Baseline (standard algorithms):  {baseline:>12.0f}")
print(f"  GA best found:                   {best:>12.0f}")
print(f"  Optimal (NetworkX):              {optimal:>12.0f}")
print()

print("Quality Metrics:")
print(f"  GA vs Baseline:                  {improvement:>12.1f}%")
print(f"  GA vs Optimal (gap):             {gap:>12.1f}%")
print()

print("Execution Time:")
print(f"  Total time:                      {elapsed:>12.1f}s")
print(f"  Time per generation:             {elapsed/10:>12.1f}s")
print()

RESULTS SUMMARY

Fitness Comparison:
  Baseline (standard algorithms):         39422
  GA best found:                          42370
  Optimal (NetworkX):                     45337

Quality Metrics:
  GA vs Baseline:                           7.5%
  GA vs Optimal (gap):                      6.5%

Execution Time:
  Total time:                             383.8s
  Time per generation:                     38.4s



In [15]:
print("="*80)
print("COMPUTING INDIVIDUAL ALGORITHM WEIGHTS")
print("="*80)
print()

# Get individual algorithm weights using baseline vector
vector = CanonicalVector(
    luby_base_probability=0.5,
    luby_coeff_degree=0.0,
    luby_coeff_neighbors_unmatched=0.0,
    luby_coeff_clustering=0.0,
    luby_coeff_matched=0.0,
    luby_coeff_round=0.0,
    luby_coeff_weight=0.0,
    itai_timeout_rounds=5,
    max_iterations=10,
    convergence_threshold=0.05,
)

try:
    # Get Greedy weight
    print("  Computing Greedy...", end=" ", flush=True)
    greedy_param = GreedyParameterizer()
    greedy_matching = greedy_param.execute(graph, vector)
    greedy_weight = sum(graph.get_edge_weight(u, v) for u, v in greedy_matching.items() if u < v)
    print(f"✓ {greedy_weight:.0f}")
    
    # Get Itai weight
    print("  Computing Itai...", end=" ", flush=True)
    itai_param = ItaiParameterizer()
    itai_matching = itai_param.execute(graph, vector)
    itai_weight = sum(graph.get_edge_weight(u, v) for u, v in itai_matching.items() if u < v)
    print(f"✓ {itai_weight:.0f}")
    
    # Get Luby weight
    print("  Computing Luby...", end=" ", flush=True)
    luby_param = LubyParameterizer()
    luby_matching = luby_param.execute(graph, vector)
    luby_weight = sum(graph.get_edge_weight(u, v) for u, v in luby_matching.items() if u < v)
    print(f"✓ {luby_weight:.0f}")
    
except Exception as e:
    print(f"\n✗ Error computing algorithms: {e}")
    print("Using fallback values...")
    greedy_weight = baseline * 0.95
    itai_weight = baseline * 0.98
    luby_weight = baseline * 0.96

print()

# Display individual algorithm results
print("Individual Algorithm Results:")
print(f"  Greedy:                        {greedy_weight:>12.0f}")
print(f"  Itai-Israeli:                  {itai_weight:>12.0f}")
print(f"  Luby Randomized:               {luby_weight:>12.0f}")
print(f"  Merged (baseline):             {baseline:>12.0f}")
print(f"  GA Best:                       {best:>12.0f}")
print(f"  NetworkX Optimal:              {optimal:>12.0f}")
print()


COMPUTING INDIVIDUAL ALGORITHM WEIGHTS

  Computing Greedy... ✓ 16516
  Computing Itai... ✓ 35953
  Computing Luby... ✓ 35682

Individual Algorithm Results:
  Greedy:                               16516
  Itai-Israeli:                         35953
  Luby Randomized:                      35682
  Merged (baseline):                    39422
  GA Best:                              42370
  NetworkX Optimal:                     45337



## Individual Algorithm Weights

## Fitness Progression

In [ ]:
# First, run GA without cascading for comparison
print("Running GA without cascading for baseline comparison...", end=" ", flush=True)
from src.meta.core.fitness_evaluator import FitnessEvaluator
ga_standard = MetaAlgorithmGA(
    fitness_evaluator=FitnessEvaluator(),
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    mutation_rate=0.15
)
best_vector_standard, fitness_history_standard = ga_standard.evolve(graph)
print(f"✓ Done")
print(f"GA without cascading best: {fitness_history_standard[-1]:.0f}")
print(f"GA with cascading best: {fitness_history[-1]:.0f}")
print(f"Cascading improvement: {((fitness_history[-1] - fitness_history_standard[-1]) / fitness_history_standard[-1] * 100):+.1f}%")
print()

# Plot fitness progression with ALL baselines and both GA approaches
fig, ax = plt.subplots(figsize=(14, 7))

gens = list(range(1, len(fitness_history) + 1))

# Plot GA with cascading (current)
ax.plot(gens, fitness_history, 'mo-', linewidth=2.5, markersize=10, label='GA (with Cascading)', zorder=5)

# Plot GA without cascading (for comparison)
ax.plot(gens, fitness_history_standard, 'co--', linewidth=2.5, markersize=8, label='GA (without Cascading)', zorder=4)

# Add individual algorithm baselines
ax.axhline(y=greedy_weight, color='#FF9999', linestyle='--', linewidth=2, alpha=0.7, label=f'Greedy ({greedy_weight:.0f})')
ax.axhline(y=itai_weight, color='#99CCFF', linestyle='--', linewidth=2, alpha=0.7, label=f'Itai ({itai_weight:.0f})')
ax.axhline(y=luby_weight, color='#99FF99', linestyle='--', linewidth=2, alpha=0.7, label=f'Luby ({luby_weight:.0f})')

# Add merged baseline
ax.axhline(y=baseline, color='#FFCC99', linestyle=':', linewidth=2.5, alpha=0.8, label=f'Merged Baseline ({baseline:.0f})')

# Add cascading baseline
ax.axhline(y=baseline_cascading, color='#FF6699', linestyle=':', linewidth=2.5, alpha=0.8, label=f'Cascading Baseline ({baseline_cascading:.0f})')

# Add optimal line
ax.axhline(y=optimal, color='#00FF00', linestyle='-', linewidth=3, alpha=0.5, label=f'NetworkX Optimal ({optimal:.0f})')

ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Fitness (Matching Weight)', fontsize=12, fontweight='bold')
ax.set_title('GA Fitness Progression - With vs Without Cascading + All Baselines', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='best', framealpha=0.95)
ax.set_xticks(gens)

plt.tight_layout()
plt.show()

Running GA without cascading for baseline comparison... 

## Generation-by-Generation Analysis

In [ ]:
print("="*80)
print("GENERATION-BY-GENERATION ANALYSIS")
print("="*80)
print()

improvements = [0.0] + [fitness_history[i] - fitness_history[i-1] for i in range(1, len(fitness_history))]

print(f"{'Gen':>3} {'Fitness':>12} {'Change':>12} {'% Change':>12} {'Gap to Opt':>12}")
print("-" * 65)

for gen, fitness in enumerate(fitness_history, 1):
    change = improvements[gen-1]
    pct_change = (change / (fitness_history[0] + 1e-10)) * 100
    gap_to_opt = ((optimal - fitness) / (optimal + 1e-10)) * 100
    print(f"{gen:>3} {fitness:>12.0f} {change:>12.0f} {pct_change:>11.2f}% {gap_to_opt:>11.2f}%")

print("-" * 65)
total_improvement = fitness_history[-1] - fitness_history[0]
total_pct = (total_improvement / (fitness_history[0] + 1e-10)) * 100
print(f"Total improvement: {total_improvement:.0f} ({total_pct:+.2f}%)")
print()

## Best Parameters Found

In [ ]:
# Create comparison visualization with both baselines
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fitness comparison (all three)
values = [baseline, baseline_cascading, best, optimal]
labels = [f'Standard\n({baseline:.0f})', f'Cascading\n({baseline_cascading:.0f})', f'GA Best\n({best:.0f})', f'Optimal\n({optimal:.0f})']
colors = ['orange', 'red', 'blue', 'green']

axes[0].bar(labels, values, color=colors, alpha=0.7, width=0.6)
axes[0].set_ylabel('Matching Weight', fontweight='bold', fontsize=11)
axes[0].set_title('Fitness Comparison: Baselines vs GA', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[0].text(i, v + 100, f"{v:.0f}", ha='center', fontweight='bold', fontsize=10)

# Plot 2: Quality metrics (GA vs both baselines)
metrics = ['GA vs Standard', 'GA vs Cascading', 'Gap to Optimal']
improvement_cascading = ((best - baseline_cascading) / baseline_cascading * 100)
values_metrics = [improvement, improvement_cascading, gap]
colors_metrics = ['green' if v > 0 else 'red' for v in values_metrics]

bars = axes[1].bar(metrics, values_metrics, color=colors_metrics, alpha=0.7, width=0.6)
axes[1].set_ylabel('Percentage (%)', fontweight='bold', fontsize=11)
axes[1].set_title('Quality Metrics', fontweight='bold', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values_metrics):
    offset = 0.5 if v > 0 else -0.5
    axes[1].text(i, v + offset, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## Performance Visualization

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fitness comparison
values = [baseline, best, optimal]
labels = [f'Baseline\n({baseline:.0f})', f'GA Best\n({best:.0f})', f'Optimal\n({optimal:.0f})']
colors = ['red', 'blue', 'green']

axes[0].bar(labels, values, color=colors, alpha=0.7, width=0.6)
axes[0].set_ylabel('Matching Weight', fontweight='bold', fontsize=11)
axes[0].set_title('Fitness Comparison', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[0].text(i, v + 100, f"{v:.0f}", ha='center', fontweight='bold', fontsize=10)

# Plot 2: Quality metrics
metrics = ['GA vs Baseline', 'Gap to Optimal']
values_metrics = [improvement, gap]
colors_metrics = ['green' if v > 0 else 'red' for v in values_metrics]

bars = axes[1].bar(metrics, values_metrics, color=colors_metrics, alpha=0.7, width=0.6)
axes[1].set_ylabel('Percentage (%)', fontweight='bold', fontsize=11)
axes[1].set_title('Quality Metrics', fontweight='bold', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values_metrics):
    offset = 0.5 if v > 0 else -0.5
    axes[1].text(i, v + offset, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## Statistics Summary

In [ ]:
print("="*80)
print("ALGORITHM WEIGHT EVOLUTION COMPARISON")
print("="*80)
print()

# Get individual algorithm weights
print("Computing individual algorithm weights...", flush=True)

vector = CanonicalVector(
    luby_base_probability=0.5,
    luby_coeff_degree=0.0,
    luby_coeff_neighbors_unmatched=0.0,
    luby_coeff_clustering=0.0,
    luby_coeff_matched=0.0,
    luby_coeff_round=0.0,
    luby_coeff_weight=0.0,
    itai_timeout_rounds=5,
    max_iterations=10,
    convergence_threshold=0.05,
)

try:
    # Get Greedy weight
    print("  Computing Greedy...", end=" ", flush=True)
    greedy_param = GreedyParameterizer()
    greedy_matching = greedy_param.execute(graph, vector)
    greedy_weight = sum(graph.get_edge_weight(u, v) for u, v in greedy_matching.items() if u < v)
    print(f"✓ {greedy_weight:.0f}")
    
    # Get Itai weight
    print("  Computing Itai...", end=" ", flush=True)
    itai_param = ItaiParameterizer()
    itai_matching = itai_param.execute(graph, vector)
    itai_weight = sum(graph.get_edge_weight(u, v) for u, v in itai_matching.items() if u < v)
    print(f"✓ {itai_weight:.0f}")
    
    # Get Luby weight
    print("  Computing Luby...", end=" ", flush=True)
    luby_param = LubyParameterizer()
    luby_matching = luby_param.execute(graph, vector)
    luby_weight = sum(graph.get_edge_weight(u, v) for u, v in luby_matching.items() if u < v)
    print(f"✓ {luby_weight:.0f}")
    
except Exception as e:
    print(f"\n✗ Error computing algorithms: {e}")
    print("Using baseline values as fallback...")
    greedy_weight = baseline * 0.95
    itai_weight = baseline * 0.98
    luby_weight = baseline * 0.96

print()

# Display algorithm weights
print("Individual Algorithm Results:")
print(f"  Greedy:                        {greedy_weight:>12.0f}")
print(f"  Itai-Israeli:                  {itai_weight:>12.0f}")
print(f"  Luby Randomized:               {luby_weight:>12.0f}")
print(f"  Merged (baseline):             {baseline:>12.0f}")
print(f"  GA Best:                       {best:>12.0f}")
print(f"  NetworkX Optimal:              {optimal:>12.0f}")
print()

# Calculate improvements relative to NetworkX optimal
greedy_gap = ((optimal - greedy_weight) / (optimal + 1e-10)) * 100
itai_gap = ((optimal - itai_weight) / (optimal + 1e-10)) * 100
luby_gap = ((optimal - luby_weight) / (optimal + 1e-10)) * 100
baseline_gap = ((optimal - baseline) / (optimal + 1e-10)) * 100
ga_gap = ((optimal - best) / (optimal + 1e-10)) * 100

print("Gap to Optimal (NetworkX):")
print(f"  Greedy:                        {greedy_gap:>12.2f}%")
print(f"  Itai-Israeli:                  {itai_gap:>12.2f}%")
print(f"  Luby Randomized:               {luby_gap:>12.2f}%")
print(f"  Merged (baseline):             {baseline_gap:>12.2f}%")
print(f"  GA Best:                       {ga_gap:>12.2f}%")
print()

In [ ]:
# Create comprehensive comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Individual algorithm weights vs baselines
ax = axes[0, 0]
algorithms = ['Greedy', 'Itai', 'Luby', 'Merged\n(Baseline)', 'GA Best', 'NetworkX\nOptimal']
weights = [greedy_weight, itai_weight, luby_weight, baseline, best, optimal]
colors_alg = ['#FF9999', '#99CCFF', '#99FF99', '#FFCC99', '#FF99FF', '#00FF00']

bars = ax.bar(algorithms, weights, color=colors_alg, alpha=0.8, width=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Matching Weight', fontweight='bold', fontsize=11)
ax.set_title('Algorithm Results Comparison', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, weight) in enumerate(zip(bars, weights)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{weight:.0f}',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# Plot 2: Gap to Optimal
ax = axes[0, 1]
gaps = [greedy_gap, itai_gap, luby_gap, baseline_gap, ga_gap]
gap_labels = ['Greedy', 'Itai', 'Luby', 'Merged', 'GA Best']
colors_gap = ['#FF9999', '#99CCFF', '#99FF99', '#FFCC99', '#FF99FF']

bars = ax.bar(gap_labels, gaps, color=colors_gap, alpha=0.8, width=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Gap to Optimal (%)', fontweight='bold', fontsize=11)
ax.set_title('Distance from NetworkX Optimal', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, gap in zip(bars, gaps):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# Plot 3: Weight evolution across GA generations (with algorithm baselines)
ax = axes[1, 0]
gens = list(range(1, len(fitness_history) + 1))

# Plot GA progression
ax.plot(gens, fitness_history, 'mo-', linewidth=2.5, markersize=10, label='GA Fitness', zorder=5)

# Add algorithm baselines
ax.axhline(y=greedy_weight, color='#FF9999', linestyle='--', linewidth=2, alpha=0.7, label=f'Greedy ({greedy_weight:.0f})')
ax.axhline(y=itai_weight, color='#99CCFF', linestyle='--', linewidth=2, alpha=0.7, label=f'Itai ({itai_weight:.0f})')
ax.axhline(y=luby_weight, color='#99FF99', linestyle='--', linewidth=2, alpha=0.7, label=f'Luby ({luby_weight:.0f})')
ax.axhline(y=baseline, color='#FFCC99', linestyle=':', linewidth=2.5, alpha=0.8, label=f'Merged Baseline ({baseline:.0f})')
ax.axhline(y=optimal, color='#00FF00', linestyle='-', linewidth=3, alpha=0.5, label=f'NetworkX Optimal ({optimal:.0f})')

ax.set_xlabel('Generation', fontsize=11, fontweight='bold')
ax.set_ylabel('Matching Weight', fontsize=11, fontweight='bold')
ax.set_title('Weight Evolution: GA vs Individual Algorithms', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc='best', framealpha=0.9)
ax.set_xticks(gens)

# Plot 4: Normalized performance comparison (% relative to optimal)
ax = axes[1, 1]
performance_pct = [(w / optimal) * 100 for w in weights]
labels_perf = ['Greedy', 'Itai', 'Luby', 'Merged', 'GA Best', 'Optimal']
colors_perf = ['#FF9999', '#99CCFF', '#99FF99', '#FFCC99', '#FF99FF', '#00FF00']

bars = ax.bar(labels_perf, performance_pct, color=colors_perf, alpha=0.8, width=0.7, edgecolor='black', linewidth=1.5)
ax.axhline(y=100, color='green', linestyle='-', linewidth=2, alpha=0.5, label='Optimal (100%)')
ax.set_ylabel('% of Optimal Weight', fontweight='bold', fontsize=11)
ax.set_title('Performance as % of NetworkX Optimal', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 105])

# Add value labels
for bar, pct in zip(bars, performance_pct):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{pct:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()